# Text Sparse Representations

## Overview

Before neural networks dominated NLP, the dominant paradigm for representing text was **sparse, count-based methods**. These methods encode documents as high-dimensional vectors where each dimension corresponds to a vocabulary word and its value reflects how often (or how importantly) that word appears.  These are commonly referred to bag-of-words representation and text sparse representations.

Despite their simplicity, sparse representations remain highly competitive for many tasks, are extremely fast to compute, and are easy to interpret. They are used in classic search engines (BM25), spam filters, and many production NLP systems.

This lab explores the progression from raw word counts to more sophisticated weighting schemes:

| Method | Key Idea |
|--------|----------|
| **Bag of Words (BoW)** | Count raw occurrences of each word |
| **Binary BoW** | 1 if word present, 0 otherwise |
| **TF-IDF** | Downweight common words, upweight informative ones |
| **N-grams** | Capture short word sequences (bigrams, trigrams) |
| **Character n-grams** | Language-agnostic, robust to typos |

## Learning Objectives

By the end of this lab you should be able to:

1. Build a bag-of-words vocabulary from a text corpus using `scikit-learn`
2. Understand the difference between unigrams, bigrams and character n-grams
3. Apply TF-IDF weighting and explain why it improves over raw counts
4. Use `spaCy` for linguistic analysis: lemmatization, POS tagging, NER
5. Discuss the strengths and limitations of sparse representations vs. dense embeddings

## Course Context

This lab corresponds to the **Text Representation** topic. After mastering sparse methods, the next labs (Lab01b, Lab02a) introduce dense *word embeddings* and *semantic search*, where meaning is captured geometrically in a continuous vector space.

---

## 1. Tokenization and Vocabulary

First, text needs to be converted in minimal units of text segments, the so called tokens. A token can be a word, a span of characters or of words.

A **tokenizer** (also called an *analyser* in IR terminology) performs this job. It usually includes several sub-steps:

1. **Segmentation** – split text into candidate tokens (by whitespace, punctuation, etc.)
2. **Normalization** – lowercase, remove accents, etc.
3. **Filtering** – remove stop words (highly frequent, low-information words like *"the"*, *"is"*)
4. **Stemming / Lemmatization** – reduce words to their canonical form (*"running"* → *"run"*)

After tokenization, a **vocabulary** $V = \{v_1, \ldots, v_L\}$ is built by collecting all unique tokens across the entire corpus. Each document can then be represented as a vector over this vocabulary.

### scikit-learn's CountVectorizer

The `CountVectorizer` class is a versatile tokenizer that supports:
- `analyzer='word'` or `analyzer='char'` — word-level or character-level tokens
- `ngram_range=(min, max)` — include all n-grams from length *min* to *max*
- `stop_words` — a list or set of tokens to exclude

📖 **Reference**: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html

Try uncommenting different vectorizer configurations below to see how they affect the vocabulary.

In [3]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
     'This is the first document.',
     'This document is the second document.',
     'And this is the third one.',
     'Is this the first document?',
 ]

my_stop_words = ['is', 'the']
# UNIGRAMS
vectorizer = CountVectorizer(ngram_range=(1,1), analyzer='word', stop_words = None)
#vectorizer = CountVectorizer(ngram_range=(1,1), analyzer='word', stop_words = 'english')
vectorizer = CountVectorizer(ngram_range=(1,1), analyzer='word', stop_words = my_stop_words)

# UNIGRAMS and BIGRAMS
#vectorizer = CountVectorizer(ngram_range=(1,2), analyzer='word')

# Character GRAMS
#vectorizer = CountVectorizer(ngram_range=(3,4), analyzer='char')


After calling `vectorizer.fit(corpus)`, scikit-learn scans every document and records every unique token. The resulting **vocabulary** maps each token to an integer index.

> **Design choice — Stop words**: Words like *"the"*, *"is"*, *"and"* appear in almost every document, so they carry little discriminative power. Removing them reduces the vocabulary size and can improve both speed and accuracy. However, for some tasks (e.g., authorship attribution, grammaticality detection) stop words are informative and should be kept.

> **Design choice — N-grams**: Unigrams treat words independently. Bigrams (e.g., *"New York"*, *"not good"*) capture short-range syntax and semantics. Character n-grams are useful for morphologically rich languages and are robust to spelling variants.

The vocabulary size $L$ determines the dimensionality of the resulting document vectors. Typical values range from thousands (small corpora) to millions (large web corpora).

In [4]:
vectorizer.fit(corpus)
vocabulary = vectorizer.get_feature_names_out()
print(vocabulary)

['and' 'document' 'first' 'one' 'second' 'third' 'this']


## 2. One-Hot Encoding

**One-hot encoding** is the simplest way to convert tokens into vectors. Each vocabulary token $v_i$ is represented as a vector where only dimension $i$ is 1 and all others are 0:

$$\text{one-hot}(v_i) = [0, \ldots, 0, \underbrace{1}_{i\text{-th}}, 0, \ldots, 0]$$

**Properties:**
- Dimensionality = $|V|$ (vocabulary size)
- All token vectors are **orthogonal** — no notion of similarity between words
- *"dog"* and *"puppy"* are as distant as *"dog"* and *"photosynthesis"*
- This is a key **limitation** that motivates dense *word embeddings* (more on this later)

In [3]:
X_vocab = vectorizer.transform(vocabulary)
print(X_vocab.todense())

[[1 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 1]]


One-hot encoding is used here as a foundation. In practice, document representations are built by **aggregating** the one-hot vectors of all tokens in the document (i.e., summing counts), which gives the bag-of-words representation.

## 3. Bag-of-Words Document Representation

The **bag-of-words (BoW)** model represents a document $d_i$ as a vector of token frequencies over the vocabulary:

$$\mathbf{d}_i = \bigl(\text{count}(v_1, d_i),\; \text{count}(v_2, d_i),\; \ldots,\; \text{count}(v_L, d_i)\bigr)$$

The matrix $X \in \mathbb{R}^{N \times L}$ (N documents × L vocabulary terms) is typically very **sparse** — most documents use only a small fraction of the full vocabulary, so most entries are zero. scikit-learn returns this as a compressed sparse matrix (`scipy.sparse`).

**Why "bag-of-words"?** Because word *order* is discarded. The sentences *"the dog bit the man"* and *"the man bit the dog"* have the same BoW representation. This is a major limitation for tasks requiring syntactic understanding, but sufficient for many classification and retrieval tasks.

The next example transforms the entire set of documents of our corpus onto a 4 vectors of $L$ dimensions:

In [4]:
X = vectorizer.transform(corpus)
print(X.todense())

[[0 1 1 1 0 0 1 0 1]
 [0 2 0 1 0 1 1 0 1]
 [1 0 0 1 1 0 1 1 1]
 [0 1 1 1 0 0 1 0 1]]


## 4. TF-IDF Weighting

Raw term counts have a problem: common words like *"the"* dominate the representation. **TF-IDF** (Term Frequency–Inverse Document Frequency) addresses this by combining two signals:

### Term Frequency (TF)
How often does term $v$ appear in document $d$? More occurrences → stronger signal.

$$\text{TF}(v, d) = \frac{\text{count}(v, d)}{\text{total tokens in } d}$$

### Inverse Document Frequency (IDF)
How rare is term $v$ across all $N$ documents? Rarer terms carry more information.

$$\text{IDF}(v) = \log \frac{N}{|\{d : v \in d\}|}$$

A term appearing in every document has IDF ≈ 0 (it discriminates nothing). A term in only 1 document has high IDF (it's very specific).

### TF-IDF Score
$$\text{TF-IDF}(v, d) = \text{TF}(v, d) \times \text{IDF}(v)$$

TF-IDF is the backbone of classic information retrieval systems. The **BM25** ranking function is a probabilistic refinement of TF-IDF.

In [5]:
from sklearn.feature_extraction.text import TfidfTransformer
import numpy as np

tfidf_transformer = TfidfTransformer(smooth_idf=True, use_idf=True)
X_tfidf = tfidf_transformer.fit_transform(X)

print("BoW matrix (raw counts):")
print(X.todense())

print("\nTF-IDF matrix (weighted counts):")
np.set_printoptions(precision=3, suppress=True)
print(X_tfidf.todense())

print("\nIDF values per term (higher = rarer across documents):")
for term, idf in zip(vocabulary, tfidf_transformer.idf_):
    print(f"  {term:<12} idf = {idf:.3f}")

BoW matrix (raw counts):
[[0 1 1 1 0 0 1 0 1]
 [0 2 0 1 0 1 1 0 1]
 [1 0 0 1 1 0 1 1 1]
 [0 1 1 1 0 0 1 0 1]]

TF-IDF matrix (weighted counts):
[[0.    0.47  0.58  0.384 0.    0.    0.384 0.    0.384]
 [0.    0.688 0.    0.281 0.    0.539 0.281 0.    0.281]
 [0.512 0.    0.    0.267 0.512 0.    0.267 0.512 0.267]
 [0.    0.47  0.58  0.384 0.    0.    0.384 0.    0.384]]

IDF values per term (higher = rarer across documents):
  and          idf = 1.916
  document     idf = 1.223
  first        idf = 1.511
  is           idf = 1.000
  one          idf = 1.916
  second       idf = 1.916
  the          idf = 1.000
  third        idf = 1.916
  this         idf = 1.000


## 5. Advanced Tokenization

scikit-learn's tokenizer handles whitespace splitting and lowercasing, but it lacks **linguistic awareness**. For more sophisticated text analysis, `spaCy` provides:

| Feature | Description | Example |
|---------|-------------|---------|
| **Lemmatization** | Map inflected form to base form | *"running"* → *"run"* |
| **POS tagging** | Part-of-speech label | *"Apple"* → `PROPN` |
| **Dependency parsing** | Syntactic relationships | subject/object roles |
| **Named Entity Recognition** | Identify real-world entities | *"Apple"* → `ORG` |

Using lemmatization instead of raw words reduces vocabulary size and groups related forms (e.g., *"studies"*, *"studied"*, *"study"* all map to *"study"*).

📖 **Reference**: https://spacy.io/usage/linguistic-features

```bash
# First-time setup (run once in your terminal):
python -m spacy download en_core_web_sm
```

### Named Entity Recognition (NER)

**Named entities** are real-world objects such as persons, organizations, locations, dates, and monetary amounts. Recognizing them is important for:
- Information extraction (e.g., *"which companies are mentioned in earnings reports?"*)
- Question answering (e.g., *"who founded Apple?"*)
- Knowledge graph construction

spaCy labels entities with types like:

| Label | Meaning | Example |
|-------|---------|---------|
| `ORG` | Organization | *"Apple"* |
| `GPE` | Geo-political entity | *"U.K."*, *"Berlin"* |
| `PERSON` | Person name | *"Barack Obama"* |
| `MONEY` | Monetary value | *"$1 billion"* |
| `DATE` | Date / time | *"last Tuesday"* |

NER can be integrated into BoW systems by replacing raw tokens with their entity types, or by adding entity mentions as additional features.

*NOTE*: If you get a `ImportError: cannot import name 'display'´` error, just downgrade your IPython version to 7.31.0: 

    $ pip install ipython==7.23.1

```bash

In [6]:
import spacy
from spacy import displacy
from pathlib import Path

nlp = spacy.load("en_core_web_sm")
doc = nlp("Apple is looking at buying U.K. startup for $1 billion")

save_figures = False

print("token".ljust(10), "lemma".ljust(10), "pos".ljust(6), "tag".ljust(6), "dep".ljust(10),
            "shape".ljust(10), "alpha", "stop")
print("------------------------------------------------------------------------------")
for token in doc:
    print(token.text.ljust(10), token.lemma_.ljust(10), token.pos_.ljust(6), token.tag_.ljust(6), token.dep_.ljust(10),
            token.shape_.ljust(10), token.is_alpha, token.is_stop)

html_dep = displacy.render(doc, style="dep", jupyter=True)

if save_figures:
    file_name = "demo-dep.html"
    output_path = Path("./images/" + file_name)
    output_path.open("w", encoding="utf-8").write(html_dep)


token      lemma      pos    tag    dep        shape      alpha stop
------------------------------------------------------------------------------
Apple      Apple      PROPN  NNP    nsubj      Xxxxx      True False
is         be         AUX    VBZ    aux        xx         True True
looking    look       VERB   VBG    ROOT       xxxx       True False
at         at         ADP    IN     prep       xx         True True
buying     buy        VERB   VBG    pcomp      xxxx       True False
U.K.       U.K.       PROPN  NNP    nsubj      X.X.       False False
startup    startup    VERB   VBD    ccomp      xxxx       True False
for        for        ADP    IN     prep       xxx        True True
$          $          SYM    $      quantmod   $          False False
1          1          NUM    CD     compound   d          False False
billion    billion    NUM    CD     pobj       xxxx       True False


## Named Entities

Let's analyze the Named Entities in our corpus using spaCy. We'll extract entities and their types.

In [8]:
import spacy
from spacy import displacy
from pathlib import Path

nlp = spacy.load("en_core_web_sm")
doc = nlp("Apple is looking at buying U.K. startup for $1 billion")

for ent in doc.ents:
    print(ent.text.ljust(12), ent.label_.ljust(10), ent.start_char, ent.end_char)

html_ent = displacy.render(doc, style="ent", jupyter=True)

if save_figures:
    file_name = "demo-ent.html"
    output_path = Path("./images/" + file_name)
    output_path.open("w", encoding="utf-8").write(html_ent)


Apple        ORG        0 5
U.K.         GPE        27 31
$1 billion   MONEY      44 54




## Exercises

1. **Stop words impact**: Re-run the vectorizer with `stop_words='english'` and compare vocabulary sizes. How much does the matrix change?

2. **N-gram effects**: Build bigram (`ngram_range=(2,2)`) and unigram+bigram (`ngram_range=(1,2)`) matrices. Do bigrams capture any information that unigrams miss?

3. **TF-IDF inspection**: In the TF-IDF matrix, identify which terms have the highest weight in each document. Do these terms seem semantically important?

4. **Cosine similarity**: Compute the cosine similarity between document 0 and document 3 in both the BoW and TF-IDF matrices. Which is more informative?
   ```python
   from sklearn.metrics.pairwise import cosine_similarity
   sim = cosine_similarity(X_tfidf[0], X_tfidf[3])
   ```

5. **spaCy lemmatization**: Use spaCy to lemmatize the corpus sentences and then build a BoW vocabulary on the lemmatized tokens. How does vocabulary size change?